# Analyse-Report
### **ZÜRICH TRAM FLOW: 01/2023 – 12/2025**


**Fokus** Wo · Wann · Warum — Verspätungen im Zürcher Tramnetz  
**Ziel** Strukturelle Muster sichtbar machen · Basis für operative Empfehlungen

**Daten-Quelle**  
* IST-Daten von opentransportdata.swiss 
* GTFS- und Meteo-Datan von data.stadt-zuerich.ch

**Daten-Umfang**  
* Exploration: ~94,4 Mio. Zeilen · 16 Tramlinien · 26 Spalten
* Analyse: ~85,4 Mio. Zeilen · 16 Tramlinien · 42 Spalten

**Herausforderungen**
* Datenmenge (3 Jahre)
* Datenqualität (Betriebsbedingt)

**Überblick** 
* Erkenntnisse 
    * Netzstruktur
    * Geografie
    * Temporalität
    * Meteorologie
    * Ereignisse
    * Infrasturkur
* Empfehlungen

---

In [ ]:
from zh_tram_flow.notebook import *
import zh_tram_flow.analytics as an

TRAIN, TEST, lf, lf_all, lf_delay, lf_clean = setup_analysis("04_insights")

## Netzstruktur

**Fahrplanwechsel Dez 2023** Grösster Netzumbau der VBZ-Geschichte · L9/L11/L13
* Netzweites Delay-Signal: +0.5s
* Veränderte Linien (L11 +5.3s) ≈ unveränderte Linien (L15 +5.2s)
* Baustellenphasen · Linienanpassungen · Netzwechsel — im Signal alles unsichtbar

**Fazit** Verspätungsmuster sind strukturell — nicht ereignisbedingt

In [ ]:
import plotly.graph_objects as go
from zh_tram_flow.config import line_color

monthly = (
    lf_delay
    .with_columns([
        pl.col("operating_date").dt.strftime("%Y-%m").alias("month"),
        pl.col("line_name").cast(pl.Utf8).alias("line"),
    ])
    .group_by(["month", "line"])
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"))
    .sort(["line", "month"])
    .collect()
    .to_pandas()
)
monthly["month"] = pd.to_datetime(monthly["month"])

fig = go.Figure()
for ln in sorted(monthly["line"].unique(), key=lambda x: int(x) if x.isdigit() else 99):
    sub = monthly[monthly["line"] == ln].sort_values("month")
    fig.add_trace(go.Scatter(
        x=sub["month"], y=sub["avg_delay"],
        name=f"L{ln}", mode="lines+markers",
        line=dict(color=line_color(ln), width=1.3),
        marker=dict(size=3),
    ))

# Fahrplanwechsel + Ende Grossbaustelle
annotations = [
    ("2023-12-10", "Fahrplanwechsel<br>Dez 2023", "bottom right"),
    ("2024-06-15", "Baustellen-Ende<br>Limmatplatz",  "top right"),
    ("2024-12-08", "Fahrplanwechsel<br>Dez 2024", "bottom right"),
]
for date, label, pos in annotations:
    fig.add_vline(
        x=pd.Timestamp(date).value / 1e6,  # ms timestamp for datetime axis
        line=dict(color="#aaaaaa", width=1.2, dash="dash"),
        annotation_text=label,
        annotation_position=pos,
        annotation_font=dict(size=10, color="#999999"),
    )

fig.update_layout(
    title="Monatliche Ø Verspätung — alle Linien 2023–2025 (kein Netzschnitt)",
    yaxis_title="Ø Ankunftsverspätung (s)",
    height=480,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()
show_df(an.table_delay_before_after_switch(lf_delay))

**OTP netzweit** 87% · Ziel VBZ: 95% (±3 Minuten)
* 71.5% aller Halte: Verspätung wächst Stop zu Stop
* 27.2% bauen Verspätung ab · 1.3% neutral
* 71.3% aller dwell_time = 0s — kein Puffer eingebaut

**Fazit** Keine Kapazitätslücke — fehlendes Fahrplan-Design

In [ ]:
import plotly.graph_objects as go

otp_stats = (
    lf_clean
    .filter(pl.col("delay_delta").is_not_null())
    .select([
        (pl.col("delay_delta") < 0).sum().alias("recovering"),
        (pl.col("delay_delta") > 0).sum().alias("growing"),
        (pl.col("delay_delta") == 0).sum().alias("neutral"),
    ])
    .collect()
)
n_total = otp_stats["recovering"][0] + otp_stats["growing"][0] + otp_stats["neutral"][0]
pct_growing    = otp_stats["growing"][0]    / n_total * 100
pct_recovering = otp_stats["recovering"][0] / n_total * 100
pct_neutral    = otp_stats["neutral"][0]    / n_total * 100

fig = go.Figure(go.Bar(
    x=["Verspätung wächst", "Verspätung sinkt", "Neutral"],
    y=[pct_growing, pct_recovering, pct_neutral],
    marker_color=["#D73027", "#4575B4", "#aaa"],
    text=[f"{pct_growing:.1f}%", f"{pct_recovering:.1f}%", f"{pct_neutral:.1f}%"],
    textposition="outside",
))
fig.update_layout(
    title="Delay Delta — Akkumulierend vs. Abnehmend (Anteil aller Halte)",
    yaxis_title="Anteil (%)",
    yaxis_range=[0, 85],
    showlegend=False,
)
fig.show()

In [ ]:
an.plot_dwell_time(lf_clean, cfg)
show_df(an.table_dwell_time_by_line(lf_clean))

In [ ]:
import plotly.graph_objects as go

line_stats = (
    lf_clean
    .group_by("line_name")
    .agg([
        pl.col("arrival_delay").mean().alias("Ø Arrival Delay"),
        pl.col("dwell_time").mean().alias("Ø Dwell Time"),
    ])
    .sort("Ø Arrival Delay", descending=True)
    .collect()
    .to_pandas()
)

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Ø Arrival Delay",
    x=line_stats["line_name"],
    y=line_stats["Ø Arrival Delay"],
    marker_color="#D73027",
))
fig.add_trace(go.Bar(
    name="Ø Dwell Time (geplanter Puffer)",
    x=line_stats["line_name"],
    y=line_stats["Ø Dwell Time"],
    marker_color="#4575B4",
))
fig.update_layout(
    barmode="group",
    title="Linien — Verspätung vs. geplanter Puffer (Dwell Time)",
    yaxis_title="Sekunden",
    height=420,
)
fig.show()
show_df(line_stats)

## Geografie

**Hotspots** Periphere Aussenkorridore — nicht zentrale Knotenpunkte
* Friedhof Enzenbühl 93.8s · Balgrist 85.2s · Leutschenbach 82.7s
* Central 48.3s (15 Linien) · Paradeplatz 48.2s (14 Linien) — unter Netzschnitt

**Problemzonen**
* Kreis 11: 68.3s · OTP 83% — kein Netzausbau 2023
* Kreis 12: 66.3s — kein Netzausbau 2023

In [ ]:
an.plot_stop_delay_map(lf_clean)

In [ ]:
import plotly.express as px

df_stops = (
    an.table_top_delay_stops(lf_clean)
    .query("n >= 50000")
    .head(20)
)

fig = px.bar(
    df_stops.sort_values("avg_delay"),
    x="avg_delay",
    y="stop_name",
    orientation="h",
    color="avg_delay",
    color_continuous_scale=[[0, "#F4A261"], [0.5, "#E05A1C"], [1, "#8B0000"]],
    title="Top 20 Haltestellen — Ø Arrival Delay (min. 50k Beobachtungen)",
    labels={"avg_delay": "Ø Delay (s)", "stop_name": ""},
    text="avg_delay",
)
fig.update_traces(texttemplate="%{text:.0f}s", textposition="outside")
fig.update_layout(coloraxis_showscale=False, height=580, margin=dict(l=200))
fig.show()
show_df(df_stops)

In [ ]:
an.plot_district_analysis(lf_clean, cfg)
show_df(an.table_district_analysis(lf_clean))

## Temporalität

**Peak** 21h · 67.9s — Abreisewellen nach Events
* Kein Morgenrush: 7h = 48.9s (unter Netzschnitt)

**Wochentag**
* Schlechtester: Donnerstag · 60.4s · P95=194s
* Beste: Sonntag 48.4s · Montag 52.3s (Homeoffice-Effekt)

In [ ]:
an.plot_hour_of_day(lf_clean, cfg)
show_df(an.table_hour_of_day(lf_clean))
an.plot_day_of_week(lf_clean, cfg)
show_df(an.table_day_of_week(lf_clean))

## Meteologie

**Stärkster Faktor** Schnee · +54s · OTP −10.9pp

**Geografische Trennung**
* Schnee → Höhenlagen: Kreise 10/4/12
* Starkregen → Flusstäler: Kreis 5 (Escher Wyss / Toni-Areal)

**Linien-Paradox**
* L17: Schnee +7.7s · Regen +41.2s
* L9: Schnee +75.9s · Regen +10.0s

In [ ]:
an.plot_weather_overview(lf_clean, cfg)
show_df(an.table_weather_overview(lf_clean))

In [ ]:
# Gleiche Farbskala für beide Karten — nur die 3-4 extremen Haltestellen sättigen auf Rot
# vmax=60: Stops mit Δ>60s = tiefrot, Stops mit Δ~20-30s = hellorange
an.plot_weather_stop_map(lf_clean, flag="has_snow",       vmax=60)
an.plot_weather_stop_map(lf_clean, flag="has_heavy_rain", vmax=60)

**Beste Jahreszeit** Winter · 51.7s · OTP 88.9%
* Frühling 55.6s · Sommer 56.4s · Herbst 61.2s (schlechteste)
* Weniger Kfz-Verkehr im Winter → weniger Kreuzungskonflikte — übertrifft Schnee-Effekt

## Ereignisse

**Feiertage** Bester Tag-Typ · 46.3s · OTP 90.6% · −9.9s vs. Normal
* Grosse Events: +10.5s — fast ausschliesslich abends 18–22h
* Tagsüber: Event-Tage ≈ Normaltage

In [ ]:
an.plot_events_overview(lf_clean, cfg)
show_df(an.table_events_overview(lf_clean))

**Schlechteste Kategorie** Fachmessen · 66.0s · OTP 84%
* Schlechtester Tag: Berufsmesse Zürich 21.11.2024 · 192.5s · OTP 54.5%
* Taylor Swift: 75.4s — Fachmessen schlagen Popkonzerte

In [ ]:
import plotly.graph_objects as go

# Täglicher Delay Delta: positiv = Verspätung akkumuliert, negativ = wird abgebaut
daily_delta = (
    lf_clean
    .filter(pl.col("delay_delta").is_not_null())
    .group_by("operating_date")
    .agg(pl.col("delay_delta").mean().alias("Delay Delta"))
    .sort("operating_date")
    .collect()
    .to_pandas()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_delta["operating_date"],
    y=daily_delta["Delay Delta"],
    name="Delay Delta",
    mode="lines",
    line=dict(width=1.2, color="#D73027"),
    fill="tozeroy",
    fillcolor="rgba(215,48,39,0.12)",
))
fig.add_hline(y=0, line=dict(color="#888888", width=1, dash="dot"))
fig.update_layout(
    title="Täglicher Delay Delta 2023–2025 (positiv = Verspätung wächst Stop zu Stop)",
    yaxis_title="Ø Delay Delta (s)",
    height=360,
    hovermode="x unified",
)
fig.show()

In [ ]:
import plotly.graph_objects as go

# Arrival vs. Departure — gemeinsam in einem Chart (ohne Ferien/Sonstiges-Marker)
daily = (
    lf_clean
    .filter(pl.col("departure_delay").is_not_null())
    .group_by("operating_date")
    .agg([
        pl.col("arrival_delay").mean().alias("Arrival Delay"),
        pl.col("departure_delay").mean().alias("Departure Delay"),
    ])
    .sort("operating_date")
    .collect()
    .to_pandas()
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily["operating_date"], y=daily["Arrival Delay"],
    name="Arrival Delay", mode="lines",
    line=dict(width=1.5, color="#D73027"),
))
fig.add_trace(go.Scatter(
    x=daily["operating_date"], y=daily["Departure Delay"],
    name="Departure Delay", mode="lines",
    line=dict(width=1.5, color="#4575B4"),
))
fig.update_layout(
    title="Täglicher Delay 2023–2025 — Arrival vs. Departure",
    yaxis_title="Ø Delay (s)",
    height=380,
    hovermode="x unified",
)
fig.show()
show_df(an.table_daily_delay_timeline(lf_clean))

## Infrastruktur

**Netzausbau 2023**
* Sihlcity (K3): 55.7s · Rehalp (K8): 63.7s — moderate Verspätung
* K11 (68.3s · OTP 83%) · K12 (66.3s) — Problemzonen · kein Ausbau

**Fazit** 0 Überschneidung: Investitionsort ≠ Problemort

In [ ]:
an.plot_service_quality_district_map(lf_delay)
show_df(an.table_service_quality_by_district(lf_delay))

In [ ]:
import plotly.graph_objects as go
import json
import numpy as np
from zh_tram_flow.config import PATHS

# Stadtkreise eingefärbt nach Ø Arrival Delay — gleicher Kartentyp wie Netzausbau-Karte
district_delay = (
    lf_clean
    .group_by("district_nr")
    .agg([
        pl.col("arrival_delay").mean().alias("avg_delay"),
        ((pl.col("arrival_delay").abs() <= 120).mean() * 100).alias("otp_pct"),
        pl.len().alias("n"),
    ])
    .collect()
    .to_pandas()
)
delay_map = dict(zip(district_delay["district_nr"].astype(str), district_delay["avg_delay"].round(1)))
otp_map   = dict(zip(district_delay["district_nr"].astype(str), district_delay["otp_pct"].round(1)))

geo_path = PATHS["raw"] / "stadtkreise.geojson"
with open(geo_path) as f:
    stadtkreise = json.load(f)

kreis_ids    = [str(feat["properties"]["objid"]) for feat in stadtkreise["features"]]
kreis_delays = [delay_map.get(kid, 0) for kid in kreis_ids]

label_lats, label_lons, label_texts = [], [], []
for feat in stadtkreise["features"]:
    coords = np.array(feat["geometry"]["coordinates"][0])
    label_lats.append(coords[:, 1].mean())
    label_lons.append(coords[:, 0].mean())
    kid = feat["properties"]["objid"]
    d   = delay_map.get(str(kid), 0)
    otp = otp_map.get(str(kid), 0)
    label_texts.append(f"K{kid}<br>{d:.0f}s / {otp:.0f}%")

fig = go.Figure()
fig.add_trace(go.Choroplethmapbox(
    geojson=stadtkreise,
    locations=kreis_ids,
    z=kreis_delays,
    featureidkey="properties.objid",
    colorscale=[[0, "#4575B4"], [0.4, "#FEE090"], [1, "#D73027"]],
    zmin=min(kreis_delays),
    zmax=max(kreis_delays),
    colorbar=dict(title="Ø Delay (s)", len=0.5, y=0.5),
    marker=dict(line=dict(color="#888888", width=1), opacity=0.7),
    hovertemplate="<b>Kreis %{location}</b><br>Ø Delay: %{z:.1f}s<extra></extra>",
))
fig.add_trace(go.Scattermapbox(
    lat=label_lats, lon=label_lons, mode="text", text=label_texts,
    textfont=dict(size=11, color="#dddddd"), hoverinfo="skip",
))
fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_zoom=11, mapbox_center={"lat": 47.378, "lon": 8.540},
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    height=600,
    title="Stadtkreise — Ø Arrival Delay (blau = pünktlich, rot = verspätet)",
)
fig.show()
show_df(district_delay.sort_values("avg_delay", ascending=False).reset_index(drop=True))

## Empfehlungen

**Strukturelle Muster** — stabil über 3 Jahre

| Priorität | Empfehlung | Basis |
|:---|:---|:---|
| 🔴 Hoch | Puffer einbauen — 10s dwell_time an Aussenkorridor-Halten | 71.5% akkumulieren · 0s geplante Standzeit |
| 🔴 Hoch | Kapazität K11/K12 erhöhen — L11 kritischste Hauptlinie (68.7s) | OTP 83% · strukturell schwächste Zone |
| 🟡 Mittel | Fachmessen-Disposition — L11 Verstärkerkurse an Berufsmesse-Tagen | Schlechtester Tag: 192.5s · OTP 54.5% |
| 🟡 Mittel | Schnee-Protokoll Höhenlagen K10/K4 — Selnau Extremfall +190.9s | L9/L12 bis +76s bei Schnee |
| 🟢 Tief | Nächster Ausbau → K11/K12 statt gut-performende Kreise | 0 Overlap Investitionsort / Problemort |